# CRYCHIC 单组别常规通讯分析

本教程面向只有一个生物学条件/组别、但可包含多个 sample 和 subject 的常规分析。默认离线小数据会完整走通输入校验、dry-run、LR availability、sender evidence、结果持久化和绘图；也可以通过环境变量换成自己的 H5AD 与 checksum-pinned CellChatDB/CellPhoneDB 资源。

**可解释范围**：单组数据可以描述 LR availability、候选 sender 分配和 receptor 支持；没有组间 contrast，因此 receiver response differential、下游 attribution、整合 `comm_strength`、p/q 值和 communication probability 都不可估计。`not_estimable` 是缺少估计设计，不是生物学零。


In [ ]:
from __future__ import annotations

import hashlib
import os
import shutil
import time
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

import crychic
from crychic.resources import (
    GeneNamespace,
    Interaction,
    MappingReport,
    ResourceBundle,
    Species,
)
from crychic.sender import SenderEvidenceParameters

SEED = 20260718
USE_SYNTHETIC = os.environ.get("CRYCHIC_SINGLE_GROUP_H5AD") is None
INPUT_H5AD = Path(
    os.environ.get("CRYCHIC_SINGLE_GROUP_H5AD", "single_group_input.h5ad")
).expanduser()
DATABASE_ROOT = Path(
    os.environ.get("CRYCHIC_DATABASE_ROOT", "resources")
).expanduser()
LR_RESOURCE = os.environ.get("CRYCHIC_LR_RESOURCE", "cellchat").lower()
OUTPUT_ROOT = Path(
    os.environ.get(
        "CRYCHIC_TUTORIAL_OUTPUT_DIR",
        "tutorial_output/single_group_communication",
    )
).expanduser()
OVERWRITE_OUTPUT = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"CRYCHIC {crychic.__version__}; output={OUTPUT_ROOT.resolve()}")


## 1. 输入数据

`adata.layers['counts']` 必须是非负、integer-like 原始计数。`sample_id` 是采样/文库单位，`subject_id` 是独立生物学重复，二者不要因为当前只有一组就混为同一统计角色；`cell_type` 标记 sender/receiver 群体，`condition` 在本教程中只有一个水平。细胞只用于 sample-level 聚合，不能当作独立重复。


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def make_single_group_adata(n_subjects: int = 4) -> ad.AnnData:
    rng = np.random.default_rng(SEED)
    genes = ("H1", "H2", "L1", "L2", "R1", "R2", "T1", "T2")
    gene_index = {gene: index for index, gene in enumerate(genes)}
    rows: list[np.ndarray] = []
    obs_rows: list[dict[str, str]] = []
    obs_names: list[str] = []
    for subject_index in range(n_subjects):
        subject_id = f"subject-{subject_index + 1:02d}"
        sample_id = f"sample-{subject_index + 1:02d}"
        profiles = {
            "Sender_A": {"L1": 30.0, "L2": 5.0},
            "Sender_B": {"L1": 5.0, "L2": 24.0},
            "Receiver": {
                "R1": 22.0,
                "R2": 18.0,
                "T1": 7.0 + subject_index,
                "T2": 5.0 + subject_index,
            },
        }
        for cell_type, profile in profiles.items():
            for cell_index in range(5):
                means = np.full(len(genes), 2.0, dtype=float)
                means[gene_index["H1"]] = 18.0
                means[gene_index["H2"]] = 14.0
                for gene, value in profile.items():
                    means[gene_index[gene]] = value
                rows.append(rng.poisson(means).astype(np.int32))
                obs_rows.append(
                    {
                        "sample_id": sample_id,
                        "subject_id": subject_id,
                        "cell_type": cell_type,
                        "condition": "baseline",
                    }
                )
                obs_names.append(
                    f"{sample_id}-{cell_type}-{cell_index:02d}"
                )
    counts = sparse.csr_matrix(np.vstack(rows), dtype=np.int32)
    result = ad.AnnData(
        X=sparse.csr_matrix(counts.shape, dtype=np.float64),
        obs=pd.DataFrame(obs_rows, index=obs_names),
        var=pd.DataFrame(index=genes),
    )
    result.layers["counts"] = counts
    return result


if USE_SYNTHETIC:
    adata = make_single_group_adata()
    data_origin = "deterministic_single_group_fixture"
    input_digest = hashlib.sha256(b"crychic-single-group-tutorial-v1").hexdigest()
else:
    if not INPUT_H5AD.is_file():
        raise FileNotFoundError(INPUT_H5AD)
    adata = ad.read_h5ad(INPUT_H5AD)
    data_origin = "environment_h5ad"
    input_digest = sha256_file(INPUT_H5AD)

print(f"data_origin={data_origin}; shape={adata.shape}")
adata.obs[["condition", "subject_id", "sample_id", "cell_type"]].value_counts().rename("n_cells")


In [ ]:
config = crychic.CrychicConfig(
    context_keys=("condition",),
    counts_layer="counts",
    sample_key="sample_id",
    subject_key="subject_id",
    cell_type_key="cell_type",
    design="~ condition",
    random_seed=SEED,
)
validated = crychic.Crychic(config).validate(adata)
validation_summary = {
    "mode": validated.report.mode.value,
    "n_cells": validated.report.n_obs,
    "n_genes": validated.report.n_vars,
    "n_samples": validated.report.n_samples,
    "n_subjects": validated.report.n_subjects,
    "reason_codes": validated.report.reason_codes,
}
validation_summary


## 2. 冻结 LR 资源

默认的两个 LR 仅用于让教程离线执行，不是生物学参考数据库。真实分析应加载物种、gene namespace、版本、checksum、license 和 citation 均已审核的完整资源；替换输入时必须同时替换本单元的 synthetic resource。


In [ ]:
def tutorial_interaction(
    interaction_id: str, ligand: str, receptor: str
) -> Interaction:
    return Interaction(
        interaction_id=interaction_id,
        source_interaction_id=interaction_id,
        ligand_name=ligand,
        receptor_name=receptor,
        ligand_subunits=(ligand,),
        receptor_subunits=(receptor,),
        ligand_is_complex=False,
        receptor_is_complex=False,
        direction="Ligand-Receptor",
        source="synthetic_single_group_tutorial",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        annotation="Synthetic tutorial contract; not a biological reference",
    )


def tutorial_lr_resource() -> ResourceBundle:
    interactions = (
        tutorial_interaction("tutorial_i1", "L1", "R1"),
        tutorial_interaction("tutorial_i2", "L2", "R2"),
    )
    return ResourceBundle(
        resource_id="crychic_single_group_tutorial_lr",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        interactions=interactions,
        mapping_report=MappingReport(
            source_rows=2, loaded_rows=2, mapped_entities=4
        ),
        manifest_digest=hashlib.sha256(
            b"crychic-single-group-tutorial-lr-v1"
        ).hexdigest(),
        source_files=("embedded_synthetic_single_group_lr",),
        license="CC0-1.0",
        citation="Synthetic CRYCHIC tutorial fixture; not a biological reference.",
    )


if USE_SYNTHETIC:
    lr_resource = tutorial_lr_resource()
    resource_origin = "embedded_synthetic_contract"
elif LR_RESOURCE == "cellchat":
    lr_resource = crychic.load_cellchat_resource(
        DATABASE_ROOT, Species.HUMAN
    )
    resource_origin = "checksum_pinned_cellchat"
elif LR_RESOURCE == "cellphonedb":
    lr_resource = crychic.load_cellphonedb_resource(DATABASE_ROOT)
    resource_origin = "checksum_pinned_cellphonedb"
else:
    raise ValueError("CRYCHIC_LR_RESOURCE must be cellchat or cellphonedb")

print(
    f"resource_origin={resource_origin}; "
    f"interactions={len(lr_resource.interactions)}"
)


## 3. Dry-run 与 availability-only 拟合

先检查资源映射和每个 cell type 的 sample/subject 支持。单一 context 会出现 `single_context_no_response_contrast`，这是预期提示，不会阻止 availability-only 分析。


In [ ]:
model = crychic.Crychic(config, resource_bundle=lr_resource)
plan = model.dry_run(
    adata,
    min_cells=2,
    min_samples_per_context=2 if USE_SYNTHETIC else 1,
    min_subjects_per_context=2 if USE_SYNTHETIC else 1,
)
stage_plan = pd.DataFrame(
    [
        {
            "stage": stage.name,
            "status": stage.status.value,
            "detail": stage.detail,
        }
        for stage in plan.stages
    ]
)
print(f"can_fit={plan.can_fit}; warnings={plan.warnings}")
stage_plan


In [ ]:
result_dir = OUTPUT_ROOT / "result"
if OVERWRITE_OUTPUT and result_dir.exists():
    shutil.rmtree(result_dir)

result = model.fit(
    adata,
    output_dir=result_dir,
    input_digest=input_digest,
    persist_edge_evidence=True,
    sender_parameters=SenderEvidenceParameters(
        min_subjects=2 if USE_SYNTHETIC else 1
    ),
    min_cells=2 if USE_SYNTHETIC else 10,
    min_samples_per_context=2 if USE_SYNTHETIC else 1,
    min_subjects_per_context=2 if USE_SYNTHETIC else 1,
    min_pooled_availability=0.0,
    max_interactions=None,
)
assert isinstance(result, crychic.CrychicResult)
result = crychic.CrychicResult.load(result_dir)
result.manifest["run_id"]


## 4. 查询、审核和绘图

排名函数在 `comm_strength` 全部不可用时自动使用 `availability`，但解释时仍应显式查看 `contrast/status/reason_code`。不要跨 receiver 合并成一个全局榜单。


In [ ]:
interactions = result.read_table("interactions")
sample_scores = result.read_table("sample_scores")
responses = result.read_table("responses")
differential = result.read_table("differential")
stages = {stage["name"]: stage for stage in result.manifest["stages"]}

assert plan.can_fit
assert set(interactions["contrast"].astype(str)) == {"availability_only"}
assert interactions["availability"].notna().all()
assert interactions["comm_strength"].isna().all()
assert interactions["comm_probability"].isna().all()
assert sample_scores["comm_strength"].isna().all()
assert responses.empty
assert stages["response"] == {
    "name": "response",
    "status": "not_estimable",
    "reason_code": "no_estimable_response_contrast",
}
for field in ("p_value", "q_value", "standard_error"):
    if field in differential:
        assert differential[field].isna().all()

status_audit = (
    interactions.groupby(
        ["mode", "status", "reason_code"],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)
status_audit


In [ ]:
ranked = result.rank_interactions(
    context={"condition": "baseline"},
    receiver="Receiver",
    contrast="availability_only",
    mode="state",
    top_n=12,
)
ranked.loc[
    :,
    [
        "sender",
        "receiver",
        "interaction_id",
        "availability",
        "assignment_weight",
        "status",
        "reason_code",
    ],
]


In [ ]:
plot_data = ranked.loc[ranked["availability"].notna()].copy()
if plot_data.empty:
    print("No estimable availability rows; inspect status_audit.")
else:
    plot_data = plot_data.sort_values("availability", kind="stable")
    plot_data["edge"] = (
        plot_data["sender"].astype(str)
        + " -> "
        + plot_data["receiver"].astype(str)
        + " | "
        + plot_data["interaction_id"].astype(str)
    )
    fig, ax = plt.subplots(
        figsize=(7.2, max(3.0, 0.42 * len(plot_data)))
    )
    colors = np.where(
        plot_data["sender"].astype(str).eq("Sender_A"),
        "#2E6F95",
        "#C75B39",
    )
    ax.barh(plot_data["edge"], plot_data["availability"], color=colors)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel("LR availability (descriptive)")
    ax.set_ylabel("")
    ax.set_title("Single-group availability-only ranking")
    fig.tight_layout()
    figure_path = OUTPUT_ROOT / "single_group_availability.png"
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"figure={figure_path}")


## 结果边界与真实数据替换

- 将 `CRYCHIC_SINGLE_GROUP_H5AD` 指向自己的 H5AD；不要把 normalized/log expression 重命名为 `counts`。
- 将 `CRYCHIC_DATABASE_ROOT` 指向 checksum-pinned 数据库，并用 `CRYCHIC_LR_RESOURCE=cellchat` 或 `cellphonedb` 选择资源。
- 单组分析的 `availability_only` 适合质量控制、候选通讯描述和后续实验假设生成；它不能验证组间变化，也不能借助 target prior 凭空产生 differential response。
- 若有两个或更多条件及足够独立 subject，请使用下一份多组教程，预先声明 contrast 并运行 subject-blocked cross-fit。
